# 07 · Análisis predictivo (clasificación)
**Proyecto final – Capstone de Ciencia de Datos Aplicada (IBM)** · Autora: Jhoselyn Vera

**Objetivo:** predecir si la primera etapa del Falcon 9 aterrizará (`Class`) con Regresión Logística, SVM,
Árbol de decisión y KNN, ajustando hiperparámetros con `GridSearchCV` (cv = 10) y comparando con los datos de prueba.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

os.makedirs("../figures", exist_ok=True)

def plot_confusion_matrix(y, y_predict, title="Matriz de confusión", fname=None):
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="viridis")
    ax.set_xlabel("Etiquetas predichas"); ax.set_ylabel("Etiquetas reales"); ax.set_title(title)
    ax.xaxis.set_ticklabels(["no aterrizó", "aterrizó"]); ax.yaxis.set_ticklabels(["no aterrizó", "aterrizó"])
    if fname:
        plt.savefig(f"../figures/{fname}", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
data = pd.read_csv(BASE + "dataset_part_2.csv")
X = pd.read_csv(BASE + "dataset_part_3.csv")
print(data.shape, X.shape)

## Tarea 1 · Variable objetivo `Y`

In [ ]:
Y = data["Class"].to_numpy()
Y[:10]

## Tarea 2 · Estandarizar `X`

In [ ]:
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)      # <- aplica realmente la estandarización
X[:2, :5]

## Tarea 3 · Separar entrenamiento y prueba (80 % / 20 %)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print("Entrenamiento:", X_train.shape, "· Prueba:", X_test.shape)

## Tarea 4 · Regresión logística

In [ ]:
parameters = {"C": [0.01, 0.1, 1], "penalty": ["l2"], "solver": ["lbfgs"]}
logreg_cv = GridSearchCV(LogisticRegression(max_iter=1000), parameters, cv=10)
logreg_cv.fit(X_train, Y_train)
print("Mejores hiperparámetros:", logreg_cv.best_params_)
print("Exactitud (validación cruzada):", logreg_cv.best_score_)

In [ ]:
print("Exactitud en prueba:", logreg_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, logreg_cv.predict(X_test), "Regresión logística", "cm_logreg.png")

## Tarea 6 · SVM

In [ ]:
parameters = {"kernel": ("linear", "rbf", "poly", "sigmoid"),
              "C": np.logspace(-3, 3, 5), "gamma": np.logspace(-3, 3, 5)}
svm_cv = GridSearchCV(SVC(), parameters, cv=10)
svm_cv.fit(X_train, Y_train)
print("Mejores hiperparámetros:", svm_cv.best_params_)
print("Exactitud (validación cruzada):", svm_cv.best_score_)

In [ ]:
print("Exactitud en prueba:", svm_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, svm_cv.predict(X_test), "SVM", "cm_svm.png")

## Tarea 8 · Árbol de decisión

In [ ]:
parameters = {"criterion": ["gini", "entropy"],
              "splitter": ["best", "random"],
              "max_depth": [2 * n for n in range(1, 10)],
              "max_features": [None, "sqrt"],      # 'auto' ya no existe en scikit-learn reciente
              "min_samples_leaf": [1, 2, 4],
              "min_samples_split": [2, 5, 10]}
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=0), parameters, cv=10)
tree_cv.fit(X_train, Y_train)
print("Mejores hiperparámetros:", tree_cv.best_params_)
print("Exactitud (validación cruzada):", tree_cv.best_score_)

In [ ]:
print("Exactitud en prueba:", tree_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, tree_cv.predict(X_test), "Árbol de decisión", "cm_tree.png")

## Tarea 10 · KNN

In [ ]:
parameters = {"n_neighbors": list(range(1, 11)),
              "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
              "p": [1, 2]}
knn_cv = GridSearchCV(KNeighborsClassifier(), parameters, cv=10)
knn_cv.fit(X_train, Y_train)
print("Mejores hiperparámetros:", knn_cv.best_params_)
print("Exactitud (validación cruzada):", knn_cv.best_score_)

In [ ]:
print("Exactitud en prueba:", knn_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, knn_cv.predict(X_test), "KNN", "cm_knn.png")

## Tarea 12 · Comparación y mejor modelo

Si varios modelos empatan en exactitud de prueba, se desempata con la exactitud de validación cruzada
(la muestra de prueba es pequeña: un acierto equivale a varios puntos porcentuales).

In [ ]:
models = {"Regresión logística": logreg_cv, "SVM": svm_cv, "Árbol de decisión": tree_cv, "KNN": knn_cv}
summary = pd.DataFrame({
    "Mejores hiperparámetros": [str(m.best_params_) for m in models.values()],
    "Exactitud CV": [m.best_score_ for m in models.values()],
    "Exactitud prueba": [m.score(X_test, Y_test) for m in models.values()],
}, index=models.keys())
display(summary.round(4))

best = summary.sort_values(["Exactitud prueba", "Exactitud CV"], ascending=False).index[0]
print("Mejor modelo (prueba, desempate por validación cruzada):", best)